In [18]:
import numpy as np
import matplotlib.pyplot as plt
import random
from qiskit import QuantumCircuit,QuantumRegister, ClassicalRegister
from qiskit.circuit import Parameter
from qiskit_ibm_runtime import Options, Session, SamplerV2 as Sampler
from qiskit.result import marginal_distribution
from qiskit.transpiler import generate_preset_pass_manager
from qiskit_aer import AerSimulator
import tkinter as tk
from tkinter import ttk

def BB84(): 
    basis = ['Z','X']  #basis set
    #Alice bit choice
    Alice_bit = random.randint(0,1)

    #Alice basis choice
    Alice_basis = basis[random.randint(0,1)]

    #Applicaiton of Alice bit and Alice basis choice on qubit
    qkd_qubits = QuantumCircuit(1,1)
    if Alice_bit ==0 and Alice_basis =='Z':
        qkd_qubits.id(0)
    elif Alice_bit == 0 and Alice_basis =='X':
        qkd_qubits.h(0)
    elif Alice_bit == 1 and Alice_basis =='Z':
        qkd_qubits.x(0)
    elif Alice_bit == 1 and Alice_basis =='X':
        qkd_qubits.x(0)
        qkd_qubits.h(0)

    #Bob basis choice
    Bob_basis = basis[random.randint(0,1)]

    #application of Bob basis on qubit
    if Bob_basis == 'X':
        qkd_qubits.h(0)
        qkd_qubits.measure(0,0)
    else:
        qkd_qubits.measure(0,0)

    #Bob bit
    backend = AerSimulator()
    pm = generate_preset_pass_manager(backend = backend, optimization_level=3)
    qc_isa = pm.run(qkd_qubits)
    sampler = Sampler(mode=backend)
    counts = sampler.run([qc_isa], shots = 1).result()[0].data.c.get_counts()
    Bob_bit = list(counts.keys())[0]

    #Camparing Alice bit and Bob bit
    if int(Alice_bit) == int(Bob_bit):
        result = f"Keep bit ({Alice_bit})"
    else:
        result = 'Discard bit'

    return (Alice_bit, Alice_basis, Bob_bit, Bob_basis, result)

def run_round():
    alice_bit, alice_basis, bob_bit, bob_basis, result = BB84()
    
    #Insert into table
    table.insert("","end", values = (alice_bit, alice_basis, bob_bit, bob_basis, result))
    
    #if keep the bit then update the label
    if "Keep bit" in result:
        sifted_key.append(str(alice_bit))
        sifted_key_label.config(text = "Sifted_key: " + "".join(sifted_key))

root = tk.Tk()
root.title("BB84 Protocol")
root.geometry("750x400")

main_frame = tk.Frame(root, bd=4,relief = "solid")
main_frame.pack(padx = 15, pady = 15, fill = 'both', expand= True)
sifted_key = []

# Button
button_frame = tk.Frame(main_frame)
button_frame.pack(pady=10)
run_button = tk.Button(button_frame, text="Run Round", font=("Arial", 14), command=run_round)
run_button.pack()

# Table
column_frame = tk.Frame(main_frame, bd = 2, relief = "solid")
column_frame.pack()
columns = ("Alice Bit", "Alice Basis","Bob bit", "Bob Basis", "Result")
table = ttk.Treeview(column_frame, columns=columns, show="headings")
for col in columns:
    table.heading(col, text=col,anchor = "center")
    table.column(col, width=150, anchor = 'a')

table.pack()

# Sifted Key Display
sifted_key_label = tk.Label(main_frame, text="Sifted Key: ", font=("Arial", 13), fg="blue")
sifted_key_label.pack(pady=10)

# Run UI
root.mainloop()